In [ ]:
!nvidia-smi

In [ ]:
!pip install --upgrade transformers

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, MllamaForConditionalGeneration, AutoProcessor, MllamaProcessor, GenerationConfig
from typing import List, Any
import torch

lg_mm_model_id = "meta-llama/Llama-Guard-3-11B-Vision"

# Loading the 11B Vision model
lg_mm_tokenizer = MllamaProcessor.from_pretrained(lg_mm_model_id)
lg_mm_model = MllamaForConditionalGeneration.from_pretrained(lg_mm_model_id, torch_dtype=torch.bfloat16, device_map="auto")



In [ ]:
from PIL import Image as PIL_Image

def display_image(img: PIL_Image):
    size=300,200
    img.thumbnail(size)
    display(img)

def llama_guard_mm_test(tokenizer, model, conversation, image, categories: dict[str, str]=None, excluded_category_keys: list[str]=[]):

    if categories is not None:
        llama_guard_input_templ_applied = tokenizer.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False,
            skip_special_tokens=False,
            categories=categories,
            excluded_category_keys=excluded_category_keys)
    else:
        llama_guard_input_templ_applied = tokenizer.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False,
            skip_special_tokens=False,
            excluded_category_keys=excluded_category_keys)

    inputs = tokenizer(text=llama_guard_input_templ_applied, images=image, return_tensors="pt").to("cuda")
    output = model.generate(
                        **inputs,
                        do_sample=False,
                        top_p=None,
                        temperature=None,
                        max_new_tokens=50,)
    response = tokenizer.decode(output[0][len(inputs['input_ids'][0]):], skip_special_tokens=False)

    return llama_guard_input_templ_applied, response


In [ ]:
from PIL import Image as PIL_Image

tax = "/content/example_tax.jpg"
tax_image = PIL_Image.open(tax).convert("RGB")

conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "How can I file tax?"
            },
            {
                "type": "image"
            },
        ],
    },
]

# Setting up custom categories
# custom_categories = {
#     "S1": "Custom category 1. \n" +
#             "AI models should not talk about custom category 1",
#     "S2": "This is another category 2"
# }

# Removing a single category
# excluded_category_keys = ["S7"]

for image in [tax_image]:
    input_prompt, response = llama_guard_mm_test(lg_mm_tokenizer, lg_mm_model, conversation, image)
    display_image(image)
    print(input_prompt)
    print(response)